In [ ]:
import folium, datetime
import numpy as np
import pandas as pd
import geopandas as gpd
import pyarrow.parquet as pq
from glob import glob
from os.path import join
from multiprocessing import Pool
from skmob.core.trajectorydataframe import TrajDataFrame
from skmob.measures import individual
from skmob.preprocessing import filtering, compression, detection, clustering

In [ ]:
base_data_path = ""
key_file = ""
daily_files = sorted(glob(join(base_data_path, "veraset*")))

In [ ]:
def get_row_groups(row_group, files):
    """ Read a specific row group across all files. """
    
    return pd.concat([pq.ParquetFile(f, metadata=m).read_row_group(row_group).to_pandas() for f, m in zip(files, meta)]) 

def get_row_group_trajectories(row_group, files):
    """ Read the row group from all files and return a skmobility Trajectory DataFrame """
    df = get_row_groups(row_group, files)
    tdf = TrajDataFrame(df.reset_index(), latitude='latitude', longitude='longitude', datetime='datetime', user_id='caid')
    
    return tdf

def get_home_location(row_group, files):
    """ Read the row group from all files and return the home location of each user. """

    tdf = get_row_group_trajectories(row_group, files)
    
    return individual.home_location(tdf, show_progress=False)

In [ ]:
meta = [pq.ParquetFile(x).metadata for x in daily_files]
row_groups = np.arange(0, 10) #57431
files = [daily_files] * len(row_groups)
args = [(rg, f) for rg, f in zip(row_groups, files)]

In [ ]:
%%time
with Pool(processes=8) as pool:
    tdf = pd.concat(pool.starmap(get_row_group_trajectories, args))
tdf.head(3)

for all users

1. filter
2. compress
3. compute stops
4. calculate trajectories to stops
5. map travel with ability to scroll through time
6. create grid of input variables with gridded output plots

########### FILTERING ##########
  
  mobility_filter:
    
    max_speed_kmh: 500.0
    include_loops: False
    speed_kmh: 5.0
    max_loop: 6
    ratio_max: 0.25

  compress:
    
    spatial_radius_km: 0.2

########### INDIVIDUAL MEASURES ##########
  
  home_location:
    
    start_night: '22:00'
    end_night: '07:00'
    show_progress: False

  jump_lengths:
    
    show_progress: False
    merge: False

In [ ]:
tdf_fltr = filtering.filter(tdf,
                            max_speed_kmh=500.,
                            include_loops=False,
                            speed_kmh=5.0,
                            max_loop=6,
                            ratio_max=0.25)

In [ ]:
tdf_fltr_comp = compression.compress(tdf_fltr,
                                     spatial_radius_km=0.2)

In [ ]:
gdf_fltr_comp = gpd.GeoDataFrame(tdf_fltr_comp,
                                 geometry=gpd.points_from_xy(tdf_fltr_comp.lng,
                                                             tdf_fltr_comp.lat),
                                 crs="EPSG:4326")

In [ ]:
gdf_subset = gdf_fltr_comp.sample(300)
lat_mean = gdf_subset['lat'].mean()
lng_mean = gdf_subset['lng'].mean()

In [ ]:
gdf_subset = gdf_subset.assign(time=gdf_subset["datetime"].dt.strftime('%Y-%m-%dT%H:%M:%SZ'))
gdf_subset = gdf_subset.assign(icon="circle")
gdf_subset = gdf_subset.assign(iconstyle={"color": "green",
                                          "weight": 1,
                                          "fillColor": "green",
                                          "fillOpacity": 0.6,
                                          "radius": 13})

In [ ]:
gdf_subset = gdf_subset[['time', 'horizontal_accuracy', 'geometry', 'icon', 'iconstyle']].to_json(drop_id=True)

duration: ISO 8601 duration format (https://www.digi.com/resources/documentation/digidocs/90001488-13/reference/r_iso_8601_duration_format.htm)

In [ ]:
m = folium.Map(location=[lat_mean, lng_mean],
               zoom_start=6)

folium.plugins.TimestampedGeoJson(gdf_subset,
                                  date_options="YYYY-MM-DD HH:MM",
                                  duration="P1D",
                                  transition_time=100,
                                  ).add_to(m)
m

![title](img/dots.png)